In [1]:
# B0 — Install TF 2.20
!pip install -q --no-deps /kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels/tensorboard-2.20.0-py3-none-any.whl
!pip install -q --no-deps /kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

In [2]:
# B1 — Imports and config
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import gc
import re
import time
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
import tensorflow as tf

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
tf.experimental.numpy.experimental_enable_numpy_behavior()

_WALL_START = time.time()

BASE = Path("/kaggle/input/competitions/birdclef-2026")
MODEL_DIR = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")

ARTIFACT_PATH = Path("/kaggle/input/notebooks/tiantanghuaxiao/bridclef-2026-offline/offline_artifacts_v5_temporal12.pkl")

SR = 32000
WINDOW_SEC = 5
WINDOW_SAMPLES = SR * WINDOW_SEC
FILE_SAMPLES = 60 * SR
N_WINDOWS = 12

CFG = {
    "batch_files": 24,   # 如果 OOM 就改 32
    "proxy_reduce": "max",
    "verbose": False,
    "gc_every": 4,       # 新增
}

print("TensorFlow:", tf.__version__)
print("BASE exists:", BASE.exists())
print("MODEL_DIR exists:", MODEL_DIR.exists())
print("ARTIFACT exists:", ARTIFACT_PATH.exists())

TensorFlow: 2.20.0
BASE exists: True
MODEL_DIR exists: True
ARTIFACT exists: True


In [3]:
# B2 — Load minimal competition metadata
taxonomy = pd.read_csv(BASE / "taxonomy.csv")
sample_sub = pd.read_csv(BASE / "sample_submission.csv")

PRIMARY_LABELS = sample_sub.columns[1:].tolist()
N_CLASSES = len(PRIMARY_LABELS)
label_to_idx = {c: i for i, c in enumerate(PRIMARY_LABELS)}

FNAME_RE = re.compile(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg")

def parse_soundscape_filename(name):
    m = FNAME_RE.match(name)
    if not m:
        return {
            "file_id": None,
            "site": None,
            "date": pd.NaT,
            "time_utc": None,
            "hour_utc": -1,
            "month": -1,
        }
    file_id, site, ymd, hms = m.groups()
    dt = pd.to_datetime(ymd, format="%Y%m%d", errors="coerce")
    return {
        "file_id": file_id,
        "site": site,
        "date": dt,
        "time_utc": hms,
        "hour_utc": int(hms[:2]),
        "month": int(dt.month) if pd.notna(dt) else -1,
    }

In [4]:
# B3 — Load Perch + mapping/proxy
birdclassifier = tf.saved_model.load(str(MODEL_DIR))
infer_fn = birdclassifier.signatures["serving_default"]

bc_labels = (
    pd.read_csv(MODEL_DIR / "assets" / "labels.csv")
    .reset_index()
    .rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
)

NO_LABEL_INDEX = len(bc_labels)

taxonomy = taxonomy.copy()
taxonomy["scientific_name_lookup"] = taxonomy["scientific_name"]

bc_lookup = bc_labels.rename(columns={"scientific_name": "scientific_name_lookup"})
mapping = taxonomy.merge(
    bc_lookup[["scientific_name_lookup", "bc_index"]],
    on="scientific_name_lookup",
    how="left"
)
mapping["bc_index"] = mapping["bc_index"].fillna(NO_LABEL_INDEX).astype(int)

label_to_bc_index = mapping.set_index("primary_label")["bc_index"]
BC_INDICES = np.array([int(label_to_bc_index.loc[c]) for c in PRIMARY_LABELS], dtype=np.int32)

MAPPED_MASK = BC_INDICES != NO_LABEL_INDEX
MAPPED_POS = np.where(MAPPED_MASK)[0].astype(np.int32)
MAPPED_BC_INDICES = BC_INDICES[MAPPED_MASK].astype(np.int32)

CLASS_NAME_MAP = taxonomy.set_index("primary_label")["class_name"].to_dict()

def get_genus_hits(scientific_name):
    genus = str(scientific_name).split()[0]
    hits = bc_labels[
        bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)
    ].copy()
    return genus, hits

proxy_map = {}
unmapped_df = mapping[mapping["bc_index"] == NO_LABEL_INDEX].copy()
unmapped_non_sonotype = unmapped_df[
    ~unmapped_df["primary_label"].astype(str).str.contains("son", na=False)
].copy()

for _, row in unmapped_non_sonotype.iterrows():
    target = row["primary_label"]
    sci = row["scientific_name"]
    genus, hits = get_genus_hits(sci)
    if len(hits) > 0:
        proxy_map[target] = {
            "bc_indices": hits["bc_index"].astype(int).tolist(),
            "genus": genus,
        }

PROXY_TAXA = {"Amphibia", "Insecta", "Aves"}
SELECTED_PROXY_TARGETS = sorted([
    t for t in proxy_map.keys()
    if CLASS_NAME_MAP.get(t) in PROXY_TAXA
])

selected_proxy_pos_to_bc = {
    label_to_idx[target]: np.array(proxy_map[target]["bc_indices"], dtype=np.int32)
    for target in SELECTED_PROXY_TARGETS
}

In [5]:
# B4 — Helpers
def read_soundscape_60s(path):
    y, sr = sf.read(path, dtype="float32", always_2d=False)
    if y.ndim == 2:
        y = y.mean(axis=1)
    if sr != SR:
        raise ValueError(f"Unexpected sample rate {sr} in {path}; expected {SR}")
    if len(y) < FILE_SAMPLES:
        y = np.pad(y, (0, FILE_SAMPLES - len(y)))
    elif len(y) > FILE_SAMPLES:
        y = y[:FILE_SAMPLES]
    return y

def infer_perch_batches(paths, batch_files=24, verbose=False, proxy_reduce="max", gc_every=4):
    paths = [Path(p) for p in paths]
    n_files = len(paths)

    iterator = range(0, n_files, batch_files)
    if verbose:
        iterator = tqdm(
            iterator,
            total=(n_files + batch_files - 1) // batch_files,
            desc="Perch batches"
        )

    batch_idx = 0

    for start in iterator:
        batch_paths = paths[start:start + batch_files]
        batch_n = len(batch_paths)
        n_rows = batch_n * N_WINDOWS

        row_ids = np.empty(n_rows, dtype=object)
        filenames = np.empty(n_rows, dtype=object)
        sites = np.empty(n_rows, dtype=object)
        hours = np.empty(n_rows, dtype=np.int16)

        x = np.empty((n_rows, WINDOW_SAMPLES), dtype=np.float32)

        write_row = 0
        for path in batch_paths:
            y = read_soundscape_60s(path)
            x[write_row:write_row + N_WINDOWS] = y.reshape(N_WINDOWS, WINDOW_SAMPLES)

            meta = parse_soundscape_filename(path.name)
            stem = path.stem

            row_ids[write_row:write_row + N_WINDOWS] = [f"{stem}_{t}" for t in range(5, 65, 5)]
            filenames[write_row:write_row + N_WINDOWS] = path.name
            sites[write_row:write_row + N_WINDOWS] = meta["site"]
            hours[write_row:write_row + N_WINDOWS] = int(meta["hour_utc"])

            write_row += N_WINDOWS

        outputs = infer_fn(inputs=tf.convert_to_tensor(x))
        logits = outputs["label"].numpy().astype(np.float32, copy=False)
        emb = outputs["embedding"].numpy().astype(np.float32, copy=False)

        scores = np.zeros((n_rows, N_CLASSES), dtype=np.float32)
        scores[:, MAPPED_POS] = logits[:, MAPPED_BC_INDICES]

        for pos, bc_idx_arr in selected_proxy_pos_to_bc.items():
            sub = logits[:, bc_idx_arr]
            proxy_score = sub.max(axis=1) if proxy_reduce == "max" else sub.mean(axis=1)
            scores[:, pos] = proxy_score.astype(np.float32)

        meta_df = pd.DataFrame({
            "row_id": row_ids,
            "filename": filenames,
            "site": sites,
            "hour_utc": hours,
        })

        yield meta_df, scores, emb

        del x, outputs, logits, emb, scores, meta_df
        batch_idx += 1
        if gc_every and (batch_idx % gc_every == 0):
            gc.collect()

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))

EPS = 1e-4
def logit_clip(p, eps=EPS):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

In [6]:
# B5 — Load artifact (prior + probe + temporal-12d)
with open(ARTIFACT_PATH, "rb") as f:
    artifact = pickle.load(f)

# prior
global_prob = artifact["global_prob"]
site_lookup = {k: v for k, v in zip(artifact["site_keys"], artifact["site_vals"])}
hour_lookup = {k: v for k, v in zip(artifact["hour_keys"], artifact["hour_vals"])}
month_lookup = {k: v for k, v in zip(artifact["month_keys"], artifact["month_vals"])}
site_hour_lookup = {tuple(k): v for k, v in zip(artifact["site_hour_keys"], artifact["site_hour_vals"])}

# probe
scaler_mean = artifact["scaler_mean"]
scaler_scale = artifact["scaler_scale"]
pca_mean = artifact["pca_mean"]
pca_components = artifact["pca_components"]
active_class_idx = artifact["active_class_idx"]
coef_mat = artifact["coef_mat"]
intercept_vec = artifact["intercept_vec"]

# temporal-12d
temp_active_class_idx = artifact["temp_active_class_idx"]
temp12_coef_mat = artifact["temp12_coef_mat"]
temp12_intercept_vec = artifact["temp12_intercept_vec"]

print("artifact loaded")
print("probe classes:", active_class_idx.shape)
print("temporal-12d classes:", temp_active_class_idx.shape)

artifact loaded
probe classes: (71,)
temporal-12d classes: (71,)


In [7]:
# B6 — Test Perch inference
def build_prior_logits(meta_df):
    n = len(meta_df)
    prior_prob = np.zeros((n, N_CLASSES), dtype=np.float32)

    for i, row in meta_df.reset_index(drop=True).iterrows():
        p = global_prob.copy()

        site_vec = site_lookup.get(row["site"], None)
        hour_vec = hour_lookup.get(int(row["hour_utc"]), None)

        month_val = parse_soundscape_filename(row["filename"])["month"]
        month_vec = month_lookup.get(int(month_val), None)

        site_hour_vec = site_hour_lookup.get((row["site"], int(row["hour_utc"])), None)

        vecs = [v for v in [site_vec, hour_vec, month_vec, site_hour_vec] if v is not None]
        if len(vecs) > 0:
            p = np.vstack([p] + vecs).mean(axis=0).astype(np.float32)

        prior_prob[i] = p

    return logit_clip(prior_prob)

In [8]:
def build_prior_logits(meta_df):
    n = len(meta_df)
    prior_prob = np.zeros((n, N_CLASSES), dtype=np.float32)

    for i, row in meta_df.reset_index(drop=True).iterrows():
        p = global_prob.copy()

        site_vec = site_lookup.get(row["site"], None)
        hour_vec = hour_lookup.get(int(row["hour_utc"]), None)

        month_val = parse_soundscape_filename(row["filename"])["month"]
        month_vec = month_lookup.get(int(month_val), None)

        site_hour_vec = site_hour_lookup.get((row["site"], int(row["hour_utc"])), None)

        vecs = [v for v in [site_vec, hour_vec, month_vec, site_hour_vec] if v is not None]
        if len(vecs) > 0:
            p = np.vstack([p] + vecs).mean(axis=0).astype(np.float32)

        prior_prob[i] = p

    return logit_clip(prior_prob)

# 先保留原 V5 的第一组权重不动
ALPHA_PERCH = 0.60
ALPHA_PRIOR = 0.15
ALPHA_PROBE = 0.10
ALPHA_TEMP = 0.15

test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
if len(test_paths) == 0:
    test_paths = sorted((BASE / "train_soundscapes").glob("*.ogg"))[:20]

parts = []

for batch_meta, batch_scores_raw, batch_emb in infer_perch_batches(
    test_paths,
    batch_files=CFG["batch_files"],
    verbose=CFG["verbose"],
    proxy_reduce=CFG["proxy_reduce"],
    gc_every=CFG["gc_every"],
):
    # 1) prior logits
    batch_prior_logits = build_prior_logits(batch_meta)

    # 2) 先构建 final_scores 主体
    batch_final_scores = (
        ALPHA_PERCH * batch_scores_raw.astype(np.float32)
        + ALPHA_PRIOR * batch_prior_logits.astype(np.float32)
    )

    # 3) probe logits（只加 active classes，不建 full 234 零矩阵）
    X_test_scaled = (batch_emb.astype(np.float32) - scaler_mean[None, :]) / scaler_scale[None, :]
    X_test_centered = X_test_scaled - pca_mean[None, :]
    X_test_pca = X_test_centered @ pca_components.T

    batch_probe_logits_active = X_test_pca @ coef_mat.T + intercept_vec[None, :]
    batch_final_scores[:, active_class_idx] += ALPHA_PROBE * batch_probe_logits_active.astype(np.float32)

    # 4) temporal-12d logits（只加 temp_active classes）
    batch_n_files = len(batch_meta) // N_WINDOWS
    seq_scores = batch_scores_raw.reshape(batch_n_files, N_WINDOWS, N_CLASSES)

    # 只取 active classes，避免全类循环和全类矩阵
    seq_scores_active = np.transpose(seq_scores[:, :, temp_active_class_idx], (0, 2, 1))  # (files, active_cls, 12)

    temp_file_logits_active = (
        np.einsum("fct,ct->fc", seq_scores_active, temp12_coef_mat).astype(np.float32)
        + temp12_intercept_vec[None, :].astype(np.float32)
    )

    temp_window_logits_active = np.repeat(temp_file_logits_active, N_WINDOWS, axis=0)
    batch_final_scores[:, temp_active_class_idx] += ALPHA_TEMP * temp_window_logits_active

    # 5) probs
    batch_probs = sigmoid(batch_final_scores)

    batch_pred_df = pd.DataFrame(batch_probs, columns=PRIMARY_LABELS)
    batch_pred_df.insert(0, "row_id", batch_meta["row_id"].values)
    parts.append(batch_pred_df)

    del batch_scores_raw, batch_emb, batch_prior_logits, batch_final_scores
    del X_test_scaled, X_test_centered, X_test_pca, batch_probe_logits_active
    del seq_scores, seq_scores_active, temp_file_logits_active, temp_window_logits_active
    del batch_probs, batch_pred_df
    gc.collect()

pred_df = pd.concat(parts, ignore_index=True)
del parts
gc.collect()

I0000 00:00:1776581473.909758      66 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


0

In [9]:
# B8 — Submission
hidden_test_paths = sorted((BASE / "test_soundscapes").glob("*.ogg"))
is_hidden_test = len(hidden_test_paths) > 0

if is_hidden_test:
    submission = sample_sub[["row_id"]].merge(pred_df, on="row_id", how="left")
    submission[PRIMARY_LABELS] = submission[PRIMARY_LABELS].fillna(0.0).astype(np.float32)

    assert submission.shape == sample_sub.shape
    assert submission["row_id"].tolist() == sample_sub["row_id"].tolist()
else:
    submission = pred_df.copy()

submission.to_csv("submission.csv", index=False)

In [10]:
# B9 — Minimal diagnostics
wall_time = time.time() - _WALL_START
print("hidden_test:", is_hidden_test)
print("submission shape:", submission.shape)
print("wall time min:", round(wall_time / 60, 2))
print("nan count:", int(submission.isna().sum().sum()))

hidden_test: False
submission shape: (240, 235)
wall time min: 3.34
nan count: 0
